In [0]:
movie_df = (
    spark.read
    .table("workspace.default.movie_genre_silver")
)

display(movie_df)

In [0]:
from pyspark.sql import functions as F

# Identify the genre columns (boolean columns) in movie_df
genre_cols = [f.name for f in movie_df.schema.fields if f.dataType.simpleString() == "boolean"]

print(f"Genre columns found: {genre_cols}")

# Count how many times each genre column is True (one wide row)
agg_exprs = [F.sum(F.col(c).cast("int")).alias(c) for c in genre_cols]
wide_counts_df = movie_df.agg(*agg_exprs)

# Reshape into two columns: the genre name and its True count
genre_true_counts_df = (
    wide_counts_df
    .unpivot([], genre_cols, "genre", "count")
    .withColumn("count", F.col("count").cast("int"))
    .orderBy(F.desc("count"))
)

display(genre_true_counts_df)

In [0]:
# Count of movies released per year, with two columns: release_year and movie count
movies_per_year_df = (
    movie_df
    .groupBy("release_year")
    .count()
    .withColumnRenamed("count", "movie_count")
    .orderBy(F.desc("release_year"))
)

display(movies_per_year_df)

In [0]:
# Save both dataframes as gold tables
(
    genre_true_counts_df.write
    .mode("overwrite")
    .saveAsTable("workspace.default.genre_counts_gold")
)

(
    movies_per_year_df.write
    .mode("overwrite")
    .saveAsTable("workspace.default.movies_per_year_gold")
)

print("Gold tables saved:")
display(spark.sql("SHOW TABLES IN workspace.default"))